# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

In [2]:
# load general setup
from utils.setup_general import *

Environment loaded


Los datos de cultivos de la UPRA están procesados en el panel CEDE y también en los datos originales de la UPRA. Acá exploro ambos datos para decidir cuales uso

# Explorar Panel CEDE

In [3]:
# Cargar datos
CEDE_agricultura = pd.read_stata(
    filepath_or_buffer = RAW/'aaa_panel_CEDE/Microdatos/PANEL_AGRICULTURA_Y_TIERRA(2024).dta',
    preserve_dtypes=True)

# Reparar variables 
CEDE_agricultura['CODIGO_MUNICIPIO'] = CEDE_agricultura['codmpio'].astype("Int64").astype(str).str.zfill(5)
CEDE_agricultura['ANNO'] = CEDE_agricultura['ano'].astype("Int64")

# Desfragmentar el DF
CEDE_agricultura = CEDE_agricultura.copy()

# Conservar solo las variables de interes
vbles_aguacate = CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('aguacate')].to_list()
vbles_cafe = CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('cafe')].to_list()
vbles_id = ['codmpio', 'CODIGO_MUNICIPIO', 'ANNO']
vbles_interes = vbles_id + vbles_aguacate + vbles_cafe
CEDE_agricultura = CEDE_agricultura[vbles_interes]

In [4]:
# Organizar datos en formato long
vbles_resultado = list(CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('aguacate')])
panel_agricultura = CEDE_agricultura.melt(
    id_vars=vbles_id,
    value_vars=vbles_interes,
    value_name='valor'
)

# Mostrar información disponible de cada variable de aguacate
display(panel_agricultura.groupby(['variable', 'ANNO'])['valor'].count().unstack().transpose())

variable,ac_aguacate,ac_aguacateh,ac_aguacatenhnp,ac_aguacatepl,ac_cafe,as_aguacate,as_aguacateh,as_aguacatenhnp,as_aguacatepl,as_cafe,p_aguacate,p_aguacateh,p_aguacatenhnp,p_aguacatepl,p_cafe,r_aguacate,r_aguacateh,r_aguacatenhnp,r_aguacatepl,r_cafe
ANNO,,,,,,,,,,,,,,,,,,,,
2003,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2004,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2005,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2006,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2007,199,0,0,0,591,233,0,0,0,593,199,0,0,0,591,199,0,0,0,591
2008,225,0,0,0,598,267,0,0,0,598,225,0,0,0,598,225,0,0,0,598
2009,243,0,0,0,610,287,0,0,0,614,243,0,0,0,610,243,0,0,0,610
2010,274,0,0,0,618,312,0,0,0,620,274,0,0,0,618,274,0,0,0,618
2011,307,0,0,0,625,340,0,0,0,627,307,0,0,0,625,307,0,0,0,625


# Explorar datos UPRA

## UPRA 2019-2024

In [5]:
UPRA = pd.read_excel(
    io= RAW/'aab_UPRA/20250617_BaseAgricola20192024.xlsx',
    sheet_name='BasePagina',
    skiprows=7,
    dtype={'Código Dane municipio':str, 'Año':int})
display(UPRA.head(3))

,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Desagregación cultivo,Cultivo,Ciclo del cultivo,Grupo cultivo,Subgrupo,Año,Periodo,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha),Nombre científico del cultivo,Código del cultivo,Estado físico del cultivo
0,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2019,2019,24.00,23.00,138.00,6.00,Persea americana,2040299,En fresco
1,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2020,2020,8.52,3.52,21.12,6.00,Persea americana,2040299,En fresco
2,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2021,2021,8.52,4.52,27.12,6.00,Persea americana,2040299,En fresco


In [6]:
# Mientras procesaba los datos pensé que los demás cultivos compiten directamente por
# los factores de producción para los cultivos. Por eso voy a cargar el area sembrada
# y cosechada de los demas cultivos.

# Entendiendo el cultivo del aguacate hass y el aguacate papelillo como sustitutos
# Voy a hacer el análisis por cultivo y no por desagregación de cultivo.
# Por ejemplo, hago el análisis por "Aguacate" y no por "Aguacate Hass"
# El supuesto es válido en tanto variedades del mismo cultivo usen los
# factores de producción con intensidades similares

# Mostrar los cultivos disponibles en la base de datos

# ciclo del cultivo
print("\nCiclo\n", sorted(UPRA['Ciclo del cultivo'].unique()))
print(len(sorted(UPRA['Ciclo del cultivo'].unique())))

# Grupos de cultivos
print("\nGrupo\n", sorted(UPRA['Grupo cultivo'].unique()))
print(len(sorted(UPRA['Grupo cultivo'].unique())))

# Grupos de cultivos
print("\nSubgrupo\n", sorted(UPRA['Subgrupo'].unique()))
print(len(sorted(UPRA['Subgrupo'].unique())))


# Cultivos
cultivos_disponibles = sorted(UPRA['Cultivo'].unique())
print("\nCultivo\n", cultivos_disponibles)
print(len(cultivos_disponibles))


Ciclo
 ['Permanente', 'Transitorio']
2

Grupo
 ['Cereales', 'Cultivos para condimentos, bebidas medicinales y aromáticas', 'Cultivos tropicales tradicionales', 'Frutales', 'Hortalizas', 'Leguminosas', 'Oleaginosas', 'Raíces y tubérculos']
8

Subgrupo
 ['Anonáceas', 'Aromáticas', 'Aráceas', 'Caducifolios', 'Cereales', 'Condimentos', 'Cultivos para condimentos, bebidas medicinales y aromáticas', 'Cultivos tropicales tradicionales', 'Cítricos', 'Demás frutales', 'Hortalizas', 'Hortalizas de flor', 'Hortalizas de fruto', 'Hortalizas de hoja', 'Hortalizas de raíz', 'Hortalizas de tallo', 'Leguminosas', 'Medicinales', 'Mirtáceas', 'Oleaginosas', 'Pasifloráceas', 'Raíces y tubérculos', 'Solanáceas']
23

Cultivo
 ['Acelga', 'Achiote', 'Achira', 'Agraz - mortiño', 'Aguacate', 'Ahuyama', 'Ajo', 'Ajonjolí', 'Ají', 'Albahaca', 'Alcachofa', 'Algodón', 'Anón', 'Apio', 'Arazá', 'Arbol de pan o pepa del pan', 'Arracacha', 'Arroz', 'Arveja', 'Arándano', 'Asaí', 'Avena', 'Badea', 'Banano', 'Batata', 'B

## UPRA 2007-2018

In [7]:
UPRAantiguo = pd.read_excel(
    io= RAW/'aab_UPRA/Base Agrícola EVA 2007-2018_MADR.xlsx',
    sheet_name='FINAL',
    skiprows=1,
    dtype={'CÓD. MUN.':str, 'AÑO':int})
display(UPRAantiguo.head(3))

,CÓD. \nDEP.,DEPARTAMENTO,CÓD. MUN.,MUNICIPIO,GRUPO \nDE CULTIVO,SUBGRUPO \nDE CULTIVO,CULTIVO,DESAGREGACIÓN REGIONAL Y/O SISTEMA PRODUCTIVO,AÑO,PERIODO,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t),Rendimiento\n(t/ha),ESTADO FISICO PRODUCCION,NOMBRE \nCIENTIFICO,CICLO DE CULTIVO,Unnamed: 17
0,15,BOYACA,15114,BUSBANZA,HORTALIZAS,ACELGA,ACELGA,ACELGA,2006,2006B,2.00,1.00,1.00,1.00,FRUTO FRESCO,BETA VULGARIS,TRANSITORIO,NaN
1,25,CUNDINAMARCA,25754,SOACHA,HORTALIZAS,ACELGA,ACELGA,ACELGA,2006,2006B,82.00,80.00,"1,440.00",18.00,FRUTO FRESCO,BETA VULGARIS,TRANSITORIO,NaN
2,25,CUNDINAMARCA,25214,COTA,HORTALIZAS,ACELGA,ACELGA,ACELGA,2006,2006B,1.50,1.50,26.00,17.33,FRUTO FRESCO,BETA VULGARIS,TRANSITORIO,NaN


In [8]:
# Mostrar los cultivos disponibles en la base de datos

# ciclo del cultivo
print("\n Ciclo\n", sorted(UPRAantiguo['CICLO DE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['CICLO DE CULTIVO'].unique())))

# Grupos de cultivos
print("\n Grupo\n", sorted(UPRAantiguo['GRUPO \nDE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['GRUPO \nDE CULTIVO'].unique())))

# Subgrupo
print("\nSubgrupo\n", sorted(UPRAantiguo['SUBGRUPO \nDE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['SUBGRUPO \nDE CULTIVO'].unique())))

# Cultivo
print("\nCULTIVO\n", sorted(UPRAantiguo['CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['CULTIVO'].unique())))


 Ciclo
 ['ANUAL', 'PERMANENTE', 'TRANSITORIO']
3

 Grupo
 ['CEREALES', 'FIBRAS', 'FLORES Y FOLLAJES', 'FORESTALES', 'FRUTALES', 'HONGOS', 'HORTALIZAS', 'LEGUMINOSAS', 'OLEAGINOSAS', 'OTROS PERMANENTES', 'OTROS TRANSITORIOS', 'PLANTAS AROMATICAS, CONDIMENTARIAS Y MEDICINALES', 'TUBERCULOS Y PLATANOS']
13

Subgrupo
 ['ACELGA', 'ACHIRA', 'AGUACATE', 'AHUYAMA', 'AJI', 'AJO', 'AJONJOLI', 'ALCACHOFA', 'ALFALFA', 'ALGARROBO', 'ALGODON', 'ANON', 'APIO', 'ARANDANO', 'ARRACACHA', 'ARROZ', 'ARVEJA', 'AVENA', 'BANANO', 'BATATA', 'BERENJENA', 'BORE', 'BROCOLI', 'CACAO', 'CADUCIFOLIOS', 'CAFE', 'CALABACIN', 'CALABAZA', 'CAUCHO', 'CAÑA', 'CAÑA FLECHA', 'CEBADA', 'CEBOLLA', 'CENTENO', 'CHACHAFRUTO', 'CHAMPIÑON', 'CHONQUE', 'CILANTRO', 'CITRICOS', 'COCO', 'COL', 'CURUBA', 'ESPARRAGO', 'ESPARTO', 'ESPINACA', 'ESTROPAJO', 'FEIJOA', 'FIQUE', 'FLORES', 'FLORES Y FOLLAJES', 'FOLLAJES', 'FRAMBUESA', 'FRESA', 'FRIJOL', 'FRUTALES EXOTICOS', 'FRUTALES VARIOS', 'GARBANZO', 'GRANADILLA', 'GUANABANA', 'GUANDUL', 

## Armonizar taxonomías
Explorar las taxonomías de ambas bases de datos para armonizarlas

### Exportar clasificaciones para diligenciar manualmente

In [9]:
# Revisar area sembrada registrada en todos los años del Upra antiguo y de UPRA nuevo
# para crear el crosswalk y armonizar las taxonomías
# Lo hago por registro de area sembrada para poder garantizar la trazabilidad de los
# registros más importantes por area sembrada en Colombia

UPRAantiguo_crosswalk = UPRAantiguo.groupby(
    ['CICLO DE CULTIVO', 'GRUPO \nDE CULTIVO', 'CULTIVO'])['Área Sembrada\n(ha)'].sum().reset_index()
UPRAantiguo_crosswalk = UPRAantiguo_crosswalk.rename(columns={
        'CICLO DE CULTIVO': 'ciclo_original',
        'GRUPO \nDE CULTIVO': 'grupo_original',
        'CULTIVO': 'cultivo_original',
        'Área Sembrada\n(ha)': 'area_sembrada_ha'
    })
UPRAantiguo_crosswalk['año'] = "2007-2018"
UPRAantiguo_crosswalk['fuente_taxonomia'] = 'Eva antiguo'


UPRA_crosswalk = UPRA.groupby(
    ['Ciclo del cultivo', 'Grupo cultivo', 'Cultivo'])['Área sembrada (ha)'].sum().reset_index()
UPRA_crosswalk = UPRA_crosswalk.rename(columns={
        'Ciclo del cultivo': 'ciclo_original',
        'Grupo cultivo': 'grupo_original',
        'Cultivo': 'cultivo_original',
        'Área sembrada (ha)': 'area_sembrada_ha'
    })
UPRA_crosswalk['año'] = "2019-2024"
UPRA_crosswalk['fuente_taxonomia'] = 'Eva nuevo'


# Definir plantilla para diligenciar manualmente
plantilla = (
    pd.concat([UPRAantiguo_crosswalk, UPRA_crosswalk], ignore_index=True)
    .sort_values(['area_sembrada_ha'], ascending=False)
)
# Columnas para definir taxonomías armonizadas
plantilla[['ciclo_armonizado',
           'grupo_armonizado',
           'cultivo_armonizado',
          'notas']] = ""

# Exporto los datos para revisarlos manualmente y definir la armonización de las categorías
plantilla.to_excel(DATA/ 'config/1001_1_plantilla_crosswalk.xlsx', index=False)

### Armonizar taxonomías

In [ ]:
# Cargar clasificación que fue armonizada a mano
clasificacion_armonizada = pd.read_excel(DATA/"config/1001_2_crosswalk_armonizado.xlsx",
                                        sheet_name="Sheet1")
print("\nvisualizar datos")
display(clasificacion_armonizada.head(2))

# En la armonización de la taxonomía, se logró armonizar más del 98% de los datos de area sembrada
# del EVA 2007-2018 y del EVA 2019-2024
print("\nPorcentaje de area sembrada que se logró armonizar")
display(
    clasificacion_armonizada[['porcentaje acumulado armonizado  2018', 'porcentaje acumulado armonizado  2019']]
        .max().to_frame()*100)

# Conservar únicamente los cultivos que fueron armonizados
clasificacion_armonizada = clasificacion_armonizada[clasificacion_armonizada['Armonizado']==True]

# Idea general:
# En las columnas ['ciclo_original', grupo_original', 'cultivo_original'] están los nombres originales
# en las bases de datos EVA. 
# Los nombres armonizados quedaron en las columnas ['ciclo_armonizado',	'grupo_armonizado',	'cultivo_armonizado']
clasificacion_armonizada_2007_2018 = clasificacion_armonizada[clasificacion_armonizada['año']=='2007-2018']

UPRAantiguo = UPRAantiguo.rename(columns={'CULTIVO':'cultivo_original'})

clasificacion_armonizada_2007_2018 = clasificacion_armonizada_2007_2018[[
    'cultivo_original','ciclo_armonizado','grupo_armonizado','cultivo_armonizado']]

In [54]:
# Revisar duplicados por el valor en "cultivo_original"
# Cualquier duplicado debería tener los mismos valores en todas las filas
display(clasificacion_armonizada_2007_2018[clasificacion_armonizada_2007_2018.
    duplicated(keep=False, subset='cultivo_original')].sort_values(by='cultivo_original')
       )
# La inspección de los duplicados funciona sin problemas

,cultivo_original,ciclo_armonizado,grupo_armonizado,cultivo_armonizado
54,AJI,Transitorio,Hortalizas,Ají
142,AJI,Transitorio,Hortalizas,Ají
178,ALBAHACA,Permanente,"Cultivos para condimentos, bebidas medicinales...",Albahaca
138,ALBAHACA,Permanente,"Cultivos para condimentos, bebidas medicinales...",Albahaca
218,CURCUMA,Permanente,Raíces y tubérculos,Cúrcuma o azafrán
160,CURCUMA,Permanente,Raíces y tubérculos,Cúrcuma o azafrán
177,ESPARRAGO,Transitorio,Hortalizas,Espárrago
119,ESPARRAGO,Transitorio,Hortalizas,Espárrago
191,OREGANO,Permanente,"Cultivos para condimentos, bebidas medicinales...",Orégano
215,OREGANO,Permanente,"Cultivos para condimentos, bebidas medicinales...",Orégano


In [56]:
# Eliminar duplicados
clasificacion_armonizada_2007_2018 = clasificacion_armonizada_2007_2018.drop_duplicates(subset='cultivo_original')

In [68]:
# Revisar suma de valores de los indicadores antes y después del merge
print("\nResumen de todos los valores ANTES de armonizar")
antes = UPRAantiguo[['Área Sembrada\n(ha)',	'Área Cosechada\n(ha)',	'Producción\n(t)']].aggregate(['sum', 'count'])
display(antes)

# Agregar columnas de datos armonizados
UPRAantiguo_armonizado = UPRAantiguo.merge(
    clasificacion_armonizada_2007_2018,
    on='cultivo_original',
    how='right',
    validate='m:1'
)
print("\nResumen de todos los valores DESPUES de armonizar")
despues = UPRAantiguo_armonizado[['Área Sembrada\n(ha)',	'Área Cosechada\n(ha)',	'Producción\n(t)']].aggregate(['sum', 'count'])
display(despues)

print("\nPorcentaje de valores que se conservan: despues / antes")
display(100*despues/antes)


Resumen de todos los valores ANTES de armonizar


,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t)
sum,"60,334,863.76","51,807,216.06","573,927,302.13"
count,"210,847.00","210,847.00","210,847.00"



Resumen de todos los valores DESPUES de armonizar


,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t)
sum,"59,627,601.62","51,524,887.31","571,988,858.77"
count,"207,389.00","207,389.00","207,389.00"



Pprcentaje de valores que se conservan: despues / antes


,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t)
sum,98.83,99.46,99.66
count,98.36,98.36,98.36


## Extraer información de multiples cultivos
Para cada `cultivo_interes` en `lista_de_cultivos`, extraigo y organizo
1. Área Sembrada del cultivo de todos los demás cultivos (suma)
1. Área Cosechada del cultivo de todos los demás cultivos (suma)
1. Producción del cultivo de todos los demás cultivos (suma)

In [7]:
# Lista con el nombre de los cultivos de interés
lista_de_cultivos = cultivos_disponibles.copy()
cultivos_disponibles = ['Aguacate']

# DataFrame vacio para almacenar resultados
panel_cultivos = pd.DataFrame()

for cultivo_interes in tqdm(cultivos_disponibles):
    
    # Extraer datos del cultivo de interés ---------------------------------------------------------------------
    filas_cultivo_interes = UPRA['Cultivo']==cultivo_interes
    UPRA_cultivo_interes = UPRA.loc[filas_cultivo_interes]

    # Dar estructura al DF
    UPRA_cultivo_interes = UPRA_cultivo_interes.melt(
        id_vars=['Código Dane municipio', 'Año', 'Cultivo'],
        value_vars=['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)'],
        var_name=COL_VARIABLE_MEDICION,
        value_name=COL_VALOR
    )

    # organizar información del Panel
    UPRA_cultivo_interes[COL_CLASIFICACION_ECONOMETRIA] = 'Resultado' # Identificar el tipo de variable en el planteamiento econometrico
    UPRA_cultivo_interes = UPRA_cultivo_interes.rename(columns={'Cultivo':COL_VARIABLE_SUJETO})
    UPRA_cultivo_interes[COL_VARIABLE_DETALLE] = 'Todos los tipos de ' + UPRA_cultivo_interes[COL_VARIABLE_SUJETO]
    UPRA_cultivo_interes = UPRA_cultivo_interes.rename(columns={
        'Código Dane municipio':COL_ID_MUNICIPIO,
        'Año':COL_AÑO
    })
    UPRA_cultivo_interes[COL_NOMBRE_DE_VARIABLE] = ''
    UPRA_cultivo_interes[COL_VARIABLE_DESCRIPCION] = (UPRA_cultivo_interes[COL_VARIABLE_SUJETO] + ': ' +
                                                     UPRA_cultivo_interes[COL_VARIABLE_MEDICION] + ' de '  + 
                                                     UPRA_cultivo_interes[COL_VARIABLE_DETALLE])

    # Extraer datos de los cultivos diferentes al cultivo de interés ------------------------------------------
    # Calcular la suma del Area sembrada, Area cosechada y Producción para los demás cultivos dentro del municipio-año
    vbles_resultado = ['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)']
    UPRA_diferente_a_cultivo_interes = UPRA.loc[~filas_cultivo_interes].groupby(
        ['Código Dane municipio', 'Año', 'Grupo cultivo', ])[vbles_resultado].sum()
    UPRA_diferente_a_cultivo_interes = UPRA_diferente_a_cultivo_interes.reset_index()
    UPRA_diferente_a_cultivo_interes['cultivo'] = f'No_{cultivo_interes}_' + UPRA_diferente_a_cultivo_interes['Grupo cultivo']
    #UPRA_diferente_a_cultivo_interes[COL_CLASIFICACION_ECONOMETRIA] = 'Control' # Identificar el tipo de variable en el planteamiento econometrico
    
    # Dar estructura al DF
    UPRA_diferente_a_cultivo_interes = UPRA_diferente_a_cultivo_interes.melt(
        id_vars=['Código Dane municipio', 'Año', 'Grupo cultivo'],
        value_vars=['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)'],
        var_name=COL_VARIABLE_MEDICION,
        value_name='valor'
    )
    
    # organizar información del Panel
    UPRA_diferente_a_cultivo_interes[COL_VARIABLE_SUJETO] = cultivo_interes
    UPRA_diferente_a_cultivo_interes[COL_CLASIFICACION_ECONOMETRIA] = 'Control'
    UPRA_diferente_a_cultivo_interes = UPRA_diferente_a_cultivo_interes.rename(columns={
        'Código Dane municipio':COL_ID_MUNICIPIO,
        'Año':COL_AÑO,
        'Grupo cultivo': COL_VARIABLE_DETALLE
    })
    UPRA_diferente_a_cultivo_interes[COL_NOMBRE_DE_VARIABLE] = ''
    UPRA_diferente_a_cultivo_interes[COL_VARIABLE_DESCRIPCION] = (UPRA_diferente_a_cultivo_interes[COL_VARIABLE_SUJETO] + ': ' +
                                                                    UPRA_diferente_a_cultivo_interes[COL_VARIABLE_MEDICION] + ' de los demás cultivos - ' +
                                                                    UPRA_diferente_a_cultivo_interes[COL_VARIABLE_DETALLE])
    

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.81it/s]


In [8]:
UPRA_cultivo_interes[ORDEN_DF].head()

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05001,2019,,Aguacate,Área sembrada (ha),Todos los tipos de Aguacate,Aguacate: Área sembrada (ha) de Todos los tipo...,24.00,Resultado
1,05001,2020,,Aguacate,Área sembrada (ha),Todos los tipos de Aguacate,Aguacate: Área sembrada (ha) de Todos los tipo...,8.52,Resultado
2,05001,2021,,Aguacate,Área sembrada (ha),Todos los tipos de Aguacate,Aguacate: Área sembrada (ha) de Todos los tipo...,8.52,Resultado
3,05001,2022,,Aguacate,Área sembrada (ha),Todos los tipos de Aguacate,Aguacate: Área sembrada (ha) de Todos los tipo...,17.17,Resultado
4,05001,2023,,Aguacate,Área sembrada (ha),Todos los tipos de Aguacate,Aguacate: Área sembrada (ha) de Todos los tipo...,14.97,Resultado


In [9]:
UPRA_diferente_a_cultivo_interes[ORDEN_DF].head()

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05001,2019,,Aguacate,Área sembrada (ha),Cultivos tropicales tradicionales,Aguacate: Área sembrada (ha) de los demás cult...,545.00,Control
1,05001,2019,,Aguacate,Área sembrada (ha),Frutales,Aguacate: Área sembrada (ha) de los demás cult...,430.00,Control
2,05001,2019,,Aguacate,Área sembrada (ha),Hortalizas,Aguacate: Área sembrada (ha) de los demás cult...,400.00,Control
3,05001,2019,,Aguacate,Área sembrada (ha),Leguminosas,Aguacate: Área sembrada (ha) de los demás cult...,102.00,Control
4,05001,2019,,Aguacate,Área sembrada (ha),Raíces y tubérculos,Aguacate: Área sembrada (ha) de los demás cult...,100.00,Control


In [65]:
print(sorted(UPRAantiguo['CICLO DE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['CICLO DE CULTIVO'].unique())))

['ANUAL', 'PERMANENTE', 'TRANSITORIO']
3


In [62]:
print(sorted(UPRAantiguo['GRUPO \nDE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['GRUPO \nDE CULTIVO'].unique())))

['CEREALES', 'FIBRAS', 'FLORES Y FOLLAJES', 'FORESTALES', 'FRUTALES', 'HONGOS', 'HORTALIZAS', 'LEGUMINOSAS', 'OLEAGINOSAS', 'OTROS PERMANENTES', 'OTROS TRANSITORIOS', 'PLANTAS AROMATICAS, CONDIMENTARIAS Y MEDICINALES', 'TUBERCULOS Y PLATANOS']
13


In [53]:
print(sorted(UPRAantiguo['SUBGRUPO \nDE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['SUBGRUPO \nDE CULTIVO'].unique())))

['ACELGA', 'ACHIRA', 'AGUACATE', 'AHUYAMA', 'AJI', 'AJO', 'AJONJOLI', 'ALCACHOFA', 'ALFALFA', 'ALGARROBO', 'ALGODON', 'ANON', 'APIO', 'ARANDANO', 'ARRACACHA', 'ARROZ', 'ARVEJA', 'AVENA', 'BANANO', 'BATATA', 'BERENJENA', 'BORE', 'BROCOLI', 'CACAO', 'CADUCIFOLIOS', 'CAFE', 'CALABACIN', 'CALABAZA', 'CAUCHO', 'CAÑA', 'CAÑA FLECHA', 'CEBADA', 'CEBOLLA', 'CENTENO', 'CHACHAFRUTO', 'CHAMPIÑON', 'CHONQUE', 'CILANTRO', 'CITRICOS', 'COCO', 'COL', 'CURUBA', 'ESPARRAGO', 'ESPARTO', 'ESPINACA', 'ESTROPAJO', 'FEIJOA', 'FIQUE', 'FLORES', 'FLORES Y FOLLAJES', 'FOLLAJES', 'FRAMBUESA', 'FRESA', 'FRIJOL', 'FRUTALES EXOTICOS', 'FRUTALES VARIOS', 'GARBANZO', 'GRANADILLA', 'GUANABANA', 'GUANDUL', 'GUATILA', 'GUAYABA', 'HABA', 'HABICHUELA', 'HIGUERILLA', 'HORTALIZAS VARIAS', 'IRACA', 'JATROPHA', 'LECHUGA', 'LENTEJA', 'LULO', 'MAIZ', 'MAIZ FORRAJERO', 'MALANGA', 'MAMEY', 'MANGO', 'MANI', 'MARACUYA', 'MELON', 'MIMBRE', 'MORA', 'MORERA', 'NABO', 'NONI', 'NUECES', 'ORELLANA', 'PALMA AMARGA', 'PALMA DE ACEITE', 'P

In [54]:
# Mostrar los cultivos disponibles en la base de datos
cultivos_disponibles_antiguo = sorted(UPRAantiguo['CULTIVO'].unique())
print(cultivos_disponibles_antiguo)
print(len(cultivos_disponibles_antiguo))

['ACELGA', 'ACHICORIA', 'ACHIOTE', 'ACHIRA', 'AGRAZ', 'AGUACATE', 'AGUAJE', 'AHUYAMA', 'AJI', 'AJO', 'AJONJOLI', 'ALBAHACA', 'ALCACHOFA', 'ALFALFA', 'ALGARROBO', 'ALGODON', 'AMARANTO', 'ANIS', 'ANON', 'ANTURIO', 'APIO', 'ARANDANO', 'ARAZA', 'ARRACACHA', 'ARROZ', 'ARVEJA', 'ASAI', 'ASPARRAGUS', 'ASTROMELIA', 'AVENA', 'BACURI', 'BADEA', 'BANANITO', 'BANANO', 'BATATA', 'BERENJENA', 'BORE', 'BOROJO', 'BREVO', 'BROCOLI', 'CACAO', 'CADUCIFOLIOS', 'CAFE', 'CAIMO', 'CALABACIN', 'CALABAZA', 'CALENDULA', 'CANYARANA', 'CARDAMOMO', 'CAUCHO', 'CAÑA AZUCARERA', 'CAÑA FLECHA', 'CAÑA MIEL', 'CAÑA PANELERA', 'CEBADA', 'CEBOLLA DE BULBO', 'CEBOLLA DE RAMA', 'CEBOLLIN', 'CENTENO', 'CHACHAFRUTO', 'CHAMBA', 'CHAMPIÑON', 'CHIA', 'CHIRIMOYA', 'CHOLUPA', 'CHONQUE', 'CHONTADURO', 'CILANTRO', 'CIMARRON', 'CIRUELA', 'CITRICOS', 'CLAVEL', 'COCCULUS', 'COCO', 'COCONA', 'COL', 'COLIFLOR', 'COPOAZU', 'CORDELINE CINTA', 'COROZO', 'CRISANTEMO', 'CURCUMA', 'CURUBA', 'DATIL', 'DURAZNO', 'ENELDO', 'ESPARRAGO', 'ESPARTO',

In [56]:
print(sorted(UPRAantiguo['DESAGREGACIÓN REGIONAL Y/O SISTEMA PRODUCTIVO'].unique()))
print(len(sorted(UPRAantiguo['DESAGREGACIÓN REGIONAL Y/O SISTEMA PRODUCTIVO'].unique())))


['ACELGA', 'ACHICORIA', 'ACHIN', 'ACHIOTE (BIJA)', 'ACHIRA', 'AGRAZ', 'AGUACATE', 'AGUAJE', 'AHUYAMA', 'AJI', 'AJI DULCE', 'AJI TABASCO', 'AJO', 'AJONJOLI', 'ALBAHACA', 'ALCACHOFA', 'ALFALFA', 'ALGARROBO', 'ALGODON', 'AMARANTO', 'ANIS', 'ANON', 'ANTURIO', 'APIO', 'ARANDANO', 'ARAZA', 'ARRACACHA', 'ARROZ RIEGO', 'ARROZ SECANO MANUAL', 'ARROZ SECANO MECANIZADO', 'ARVEJA', 'ASAI', 'ASPARRAGUS', 'ASTROMELIA', 'AVENA', 'BACURI', 'BADEA', 'BANANITO', 'BANANO', 'BANANO EXPORTACION', 'BANANO MANZANO', 'BATATA', 'BERENJENA', 'BORE', 'BOROJO', 'BREVO', 'BROCOLI', 'CACAO', 'CACHACO', 'CADUCIFOLIOS', 'CAFE', 'CAIMO', 'CALABACIN', 'CALABAZA', 'CALENDULA', 'CANYARANA', 'CARDAMOMO', 'CARTUCHO-ASTROMELIA', 'CAUCHO', 'CAÑA AZUCARERA', 'CAÑA FLECHA', 'CAÑA MIEL', 'CAÑA PANELERA', 'CEBADA', 'CEBOLLA DE BULBO', 'CEBOLLA DE RAMA', 'CEBOLLIN', 'CENTENO', 'CHACHAFRUTO', 'CHAMBA', 'CHAMPIÑON', 'CHIA', 'CHILLANGUA', 'CHIRARAN', 'CHIRIMOYA', 'CHIRO', 'CHOLUPA', 'CHONQUE', 'CHONTADURO', 'CILANTRO', 'CIMARRON', '